In [84]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

train_data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

In [93]:
import importlib
import ml_pipeline
import mlflow

importlib.reload(ml_pipeline)

/usr/local/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


<module 'ml_pipeline' from '/work/ml_pipeline.py'>

In [86]:
!pip install shap


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [88]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp2")

2026/04/08 22:22:58 INFO mlflow.tracking.fluent: Experiment with name 'exp2' does not exist. Creating a new experiment.


<Experiment: artifact_location='/mlruns/1', creation_time=1775686978108, experiment_id='1', last_update_time=1775686978108, lifecycle_stage='active', name='exp2', tags={}, workspace='default'>

In [89]:
from ml_pipeline import MoisturePipeline, FullPipelineModel

pipe = MoisturePipeline(
    use_pca=False,
    use_diff=True,
    params={
        "verbosity": -1,
        "n_estimators": 100,
        "learning_rate": 0.05,
        "num_leaves": 64,
        "min_data_in_leaf": 5,
        "n_jobs": -1,
    }
)
rmse = pipe.fit(train_data_pl)

with mlflow.start_run():
    mlflow.log_params(pipe.params)
    #mlflow.log_param("pca_components", pipe.fe.pca.n_components)

    mlflow.log_metric("rmse", rmse)

    mlflow.pyfunc.log_model(
        name="model",
        python_model=FullPipelineModel(pipe)
    )

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/08 22:23:25 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


🏃 View run brawny-croc-784 at: http://mlflow:5000/#/experiments/1/runs/a2de936cb6e8419390990c425ec6aa96
🧪 View experiment at: http://mlflow:5000/#/experiments/1


### sharpray値を使い　2次微分は1,特定の特徴量だけが落としている　or 2.全体的に効いていない　をみる

In [64]:
# transform後のデータ
#X = pipe.fe.transform(train_data_pl)

# 特徴量カラム（学習に使ったもの）
#feature_cols = [c for c in X.columns if c not in pipe.drop_cols]

#X = X.select(feature_cols)
#y = train_data_pl["含水率"]  # ← 目的変数に合わせる

In [75]:
#!pip install shap

In [90]:
#del #show

train_cols = set(pipe.feature_cols)

test_df = pipe.fe.transform(train_data_pl)
test_cols = set(test_df.columns)

print("extra:", test_cols - train_cols)
print("missing:", train_cols - test_cols)

extra: {'樹種', 'species number', '含水率', 'sample number'}
missing: set()


In [94]:
from ml_pipeline import FeatureEngineer  # 定義している場所からimport


pipe = MoisturePipeline(use_diff=True)
pipe.fit(train_data_pl)


# SHAP可視化
pipe.fe.show_shap(train_data_pl, pipe.model)

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[LightGBM] [Fatal] The number of features in data (1558) is not the same as it was in training data (1555).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.


LightGBMError: The number of features in data (1558) is not the same as it was in training data (1555).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.


## モデルを引き落としてきて実験する

In [75]:
import mlflow

experiment_name = "exp2"  # もしくはあなたの実験名

experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=1
)

latest_run_id = runs.iloc[0]["run_id"]
print(latest_run_id)

d1e10136841f4842b13694e6af97a4d5


In [76]:
model = mlflow.pyfunc.load_model(f"runs:/{latest_run_id}/model")

In [77]:
mlflow.pyfunc.log_model(
    name="model",
    python_model=FullPipelineModel(pipe)
)

2026/04/04 21:35:53 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


In [79]:
model

mlflow.pyfunc.loaded_model:
  artifact_path: /mlruns/5/models/m-278c1a7af9f2414f8926ccc85351266f/artifacts
  flavor: mlflow.pyfunc.model
  run_id: d1e10136841f4842b13694e6af97a4d5